# Bài 45: Cơ chế chú ý

Mọi mô hình ở Phần 3 đều xử lý cấu trúc không gian thông qua tích chập: một mẫu lân cận cục bộ, cố định, được áp dụng như nhau ở mọi nơi (Bài 10). **Chú ý (attention)** là một cách khác để trộn thông tin trên một chuỗi hoặc một ảnh: thay vì một vùng lân cận cố định, mỗi vị trí tự quyết định, một cách động, những vị trí *khác* nào liên quan đến nó ngay lúc này, và trộn chúng theo tỉ lệ với mức độ liên quan đó. Bài này xây dựng cơ chế chú ý từ đầu, đối chiếu nó với cài đặt của chính PyTorch, và cho thấy cụ thể nó đang làm gì: truy xuất mềm, dựa trên nội dung.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Trực giác: trung bình có trọng số của các nguyên mẫu

Hãy xét ý tưởng chú ý ở dạng đơn giản nhất, với chỉ một truy vấn và một nhúm ứng viên.

Giả sử bạn có một tập $T$ tập hồ sơ trong một ngăn tủ. Nội dung của các tập hồ sơ là các vector **giá trị (value)** $v_1, \dots, v_T$ — hãy nghĩ về chúng như các nguyên mẫu (prototype), hay nội dung bộ nhớ: bất cứ thứ gì được trả về khi tập hồ sơ được chọn. Đi kèm mỗi giá trị là một **khóa (key)** $k_j$, một mô tả ngắn chỉ dùng để so khớp, không thuộc về bản thân nội dung — giống như nhãn trên tập hồ sơ, không phải thứ bên trong. Và bạn có một vector **truy vấn (query)** $q$, là mô tả về thứ bạn đang tìm kiếm, ở cùng dạng với các khóa để có thể so sánh với chúng.

Hãy nghĩ về truy vấn $q$ như cái nhãn bạn đang tìm, dù nó có thể chỉ khớp một phần. Ví dụ, $q$ có thể là "thẻ tín dụng" để nó khớp với nhiều tập hồ sơ chứa sao kê của tất cả các thẻ tín dụng riêng lẻ của bạn. Nhưng $q$ sẽ không khớp với các tập hồ sơ chứa sao kê tài khoản ngân hàng của bạn.

So sánh $q$ với mọi khóa bằng một tích vô hướng, $q^\top k_j$ — một điểm số tương đồng, lớn khi $q$ và $k_j$ hướng theo các hướng tương tự, nhỏ hoặc âm trong các trường hợp khác. Biến $T$ điểm số đó thành các trọng số không âm có tổng bằng 1 bằng softmax (Bài 36). Đầu ra khi đó là một **trung bình có trọng số của các nguyên mẫu**, đánh trọng số theo mức độ khóa của mỗi nguyên mẫu khớp với truy vấn:

$$\text{output} = \sum_{j=1}^{T} w_j\, v_j, \qquad (w_1, \dots, w_T) = \text{softmax}(q^\top k_1, \dots, q^\top k_T),$$

Nếu một khóa khớp tốt hơn hẳn các khóa còn lại, softmax đẩy trọng số của nó về gần 1 và đầu ra thu gọn về về cơ bản chỉ một giá trị đó — phép tra cứu "cứng" thông thường, như một trường hợp giới hạn. Nếu vài khóa khớp tốt tương đương nhau, đầu ra trộn chúng một cách mượt mà thay vì ép buộc một người thắng duy nhất.

Tuy nhiên, một chuỗi thực tế không chỉ có một truy vấn — một chuỗi gồm $T$ vị trí (từ, mảng ảnh, v.v.) mỗi vị trí tạo ra truy vấn riêng, khóa riêng và giá trị riêng. Hãy làm cho điều này chính xác bằng cách xếp $T$ vector truy vấn thành một ma trận $Q$, $T$ vector khóa thành $K$, và $T$ vector giá trị thành $V$:

$$
Q = \begin{bmatrix} -\; q_1^\top \;- \\ -\; q_2^\top \;- \\ \vdots \\ -\; q_T^\top \;- \end{bmatrix}, \qquad
K = \begin{bmatrix} -\; k_1^\top \;- \\ -\; k_2^\top \;- \\ \vdots \\ -\; k_T^\top \;- \end{bmatrix}, \qquad
V = \begin{bmatrix} -\; v_1^\top \;- \\ -\; v_2^\top \;- \\ \vdots \\ -\; v_T^\top \;- \end{bmatrix}
$$

Hãy nghĩ về $V$ như một bảng các nguyên mẫu và $Q$ cùng $K$ như một cách mềm, khả vi để quyết định mỗi nguyên mẫu nên nhận bao nhiêu trọng số — đây chính xác là ý nghĩa của "truy xuất mềm, dựa trên nội dung", được làm chính xác bên dưới.

Chạy phép tính một-truy-vấn ở trên một lần cho mỗi vị trí, cho cả $T$ vị trí đồng thời, chính xác là phép nhân ma trận. Theo định nghĩa của phép nhân ma trận, phần tử $(i, j)$ của $QK^\top$ là tích vô hướng của hàng $i$ của $Q$ với hàng $j$ của $K$ — chính xác là $q_i^\top k_j$, cùng điểm số tương đồng từ phiên bản một truy vấn ở trên, giờ được tính cho mọi truy vấn $i$ với mọi khóa $j$ cùng lúc trong một tích ma trận duy nhất. Áp dụng softmax cho mỗi hàng của ma trận $T \times T$ đó và nhân với $V$ sẽ tái tạo phép tính trung-bình-có-trọng-số-của-các-nguyên-mẫu ở trên, một cách độc lập, cho mọi hàng:

$$\text{output} = \text{softmax}(QK^\top)\, V$$

Đây là toàn bộ ý tưởng, ở dạng ma trận.

## Truy vấn, khóa và giá trị

Mảnh ghép duy nhất còn thiếu trong dạng ma trận ở trên là phép chuẩn hóa. Chia các điểm số thô cho $\sqrt{d_k}$ (căn bậc hai của số chiều khóa) trước softmax — không có nó, các tích vô hướng tăng lớn dần khi số chiều tăng, đẩy softmax vào một chế độ có gradient biến mất (Bài 37), cùng bản năng như phép co giãn khởi tạo trọng số ở Bài 33, giờ áp dụng cho điểm số chú ý thay vì giá trị kích hoạt của lớp. Với một thay đổi đó, đây là công thức được dùng trong thực tế:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

$Q$, $K$ và $V$ thực sự đến từ đâu? Mỗi vị trí bắt đầu từ cùng một vector đầu vào mà phần còn lại của mạng đã tạo ra cho nó — xếp $T$ vector đầu vào đó thành một ma trận $X$, mỗi vị trí một hàng, giống như chính $Q$, $K$, $V$. $Q$, $K$ và $V$ khi đó là ba phép chiếu tuyến tính riêng biệt của cùng $X$ đó, mỗi phép qua ma trận trọng số học được riêng của nó:

$$Q = XW_Q, \qquad K = XW_K, \qquad V = XW_V$$

$W_Q$, $W_K$ và $W_V$ được học trong khi huấn luyện (Bài 31) — hạ gradient quyết định mỗi phép chiếu nên trích xuất gì từ $X$ — và cùng ba ma trận đó được dùng lại cho mọi vị trí và mọi độ dài chuỗi, đó chính xác là điều cho phép một lớp chú ý chạy trên các chuỗi có độ dài bất kỳ. Về trực giác, $W_Q$ biến vector của một vị trí thành "tôi đang tìm gì", $W_K$ biến nó thành "tôi có gì để cung cấp, để người khác tìm kiếm", và $W_V$ biến nó thành "tôi thực sự đóng góp gì nếu được chọn" — ba cách đọc khác nhau của cùng một vector bên dưới.

Rất dễ mất dấu các hình dạng trong tất cả những điều này. $X$ có kích thước $T \times D$ — $T$ hàng, mỗi vị trí chuỗi một hàng, $D$ cột, số chiều làm việc của mô hình. $W_Q$ và $W_K$ có kích thước $D \times d_k$, và $W_V$ là $D \times d_v$, nên $Q$ và $K$ ra kích thước $T \times d_k$ và $V$ ra kích thước $T \times d_v$. Do đó $QK^\top$ có kích thước $T \times T$: một ma trận đầy đủ các điểm số tương đồng từng cặp, mỗi vị trí truy vấn một hàng, mỗi vị trí khóa một cột. Softmax chuẩn hóa mỗi hàng một cách độc lập, nên mỗi hàng vẫn có tổng bằng 1, và nhân ma trận trọng số $T \times T$ đó với các hàng $T \times d_v$ của $V$ cho ra một đầu ra $T \times d_v$ — cùng số hàng với đầu vào, mỗi hàng là một sự pha trộn có trọng số các hàng của $V$.

Để gắn các con số thực cho các chữ cái này: một Vision Transformer (Bài 47) chia một ảnh 224x224 thành các mảng 16x16, cho ra $14 \times 14 = 196$ mảng, hay $T \approx 197$ khi thêm một token phân loại, với số chiều mô hình $D$ khoảng 768 cho một mô hình cỡ "base". Chia qua, chẳng hạn, 12 đầu chú ý (attention head) (Bài 46), $d_k$ của mỗi đầu là $768 / 12 = 64$ — và $d_v$, số chiều giá trị, trong thực tế thường chỉ được đặt bằng $d_k$ (nên cũng là 64 ở đây), dù công thức thực ra không đòi hỏi điều đó: chỉ $Q$ và $K$ phải có cùng số chiều, vì tích vô hướng của chúng phải được định nghĩa, còn độ rộng của $V$ tự do là bất cứ giá trị nào tiện lợi (nó đơn giản là cũng đặt độ rộng của đầu ra). Vậy tính theo số tròn: $T$ thường cỡ hàng trăm (nhiều hơn với chuỗi văn bản dài hơn, ảnh độ phân giải cao hơn, hoặc video), $D$ cỡ hàng trăm đến vài nghìn, và $d_k$/$d_v$ của mỗi đầu thường ở mức khiêm tốn từ vài chục đến khoảng 128 — đủ nhỏ để $QK^\top$ của một ảnh, với khoảng $197 \times 197 \approx 39{,}000$ phần tử, là rẻ. Nhưng con số đó tăng theo *bình phương* của $T$, đó chính xác là lý do các chuỗi dài hơn trở nên đắt đỏ rất nhanh.

Sơ đồ bên dưới điền chính xác các con số này, với kích thước của mỗi ô được vẽ gần tỉ lệ với hình dạng thực tế của nó:

<img src="../figures/attention_pipeline_static.png" alt="Sơ đồ tĩnh của quy trình chú ý, với các ô được vẽ gần tỉ lệ với hình dạng ma trận của chúng: X kích thước T nhân D rộng và thấp; Q, K và V cao và hẹp, kích thước T nhân d_k hoặc T nhân d_v, mỗi cái được tạo ra bởi X nhân với ma trận trọng số học được riêng W chỉ số Q, W chỉ số K hoặc W chỉ số V; các ô QK-chuyển-vị và softmax vuông và tương đối lớn, kích thước T nhân T; trọng số softmax và V kết hợp trực tiếp vào ô Output, cũng cao và hẹp, kích thước T nhân d_v" style="display: block; margin: 0 auto; max-width: 980px;">

In [ ]:
def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / np.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

torch.manual_seed(0)
B, T, D = 2, 5, 8  # batch, sequence length, dimension
Q = torch.randn(B, T, D)
K = torch.randn(B, T, D)
V = torch.randn(B, T, D)

out, weights = attention(Q, K, V)
out_torch = F.scaled_dot_product_attention(Q, K, V)

print(f'output shape: {tuple(out.shape)}')
print(f'each attention-weight row sums to 1: {torch.allclose(weights.sum(-1), torch.ones(B, T))}')
print(f'max abs diff vs F.scaled_dot_product_attention: {(out - out_torch).abs().max().item():.2e}')

Ảnh động bên dưới đi qua công thức từng hàng truy vấn một cho một ví dụ nhỏ ($T=5$, $D=6$): chọn một hàng truy vấn, chấm điểm nó với mọi hàng khóa, biến các điểm số đó thành trọng số softmax, và dùng các trọng số đó để trộn các hàng của $V$ thành một hàng đầu ra — lặp lại cho mọi vị trí cho đến khi ma trận đầu ra đầy đủ được điền xong.

<img src="../animations/attention_qkv_demo.gif" alt="Ảnh động tính toán từng bước cơ chế chú ý cho một ví dụ Q, K, V nhỏ (T=5, D=6), mỗi lần một hàng truy vấn: với mỗi hàng i, hàng i của Q được làm nổi bật, rồi các điểm số tích vô hướng thô với mọi hàng khóa được hiển thị thành một thanh 1 nhân T có nhãn, rồi softmax biến các điểm số đó thành trọng số có tổng bằng 1, được trực quan hóa bằng cách tô màu các hàng của V theo tỉ lệ trọng số của chúng, và hàng đầu ra tổng có trọng số thu được được điền vào một ma trận đầu ra T nhân D lớn dần" style="display: block; margin: 0 auto; max-width: 760px;">

## Chú ý như truy xuất mềm, dựa trên nội dung

Gác công thức trừu tượng sang một bên, đây là điều chú ý thực sự làm một cách cụ thể: cho một truy vấn, nó tra cứu các khóa giống nhất trong chuỗi và trả về một sự pha trộn các giá trị của chúng, đánh trọng số theo độ tương đồng. Hãy dựng một chuỗi đồ chơi gồm 5 "token" riêng biệt (các vector one-hot, được phóng lớn để làm khác biệt tương đồng sắc nét hơn) và truy vấn token khớp chính xác với vị trí 2.

In [ ]:
vocab = torch.eye(6) * 5.0  # 6 distinct tokens, scaled to sharpen dot-product contrast
seq = vocab[[0, 1, 2, 3, 4]].unsqueeze(0)     # a 5-token sequence: tokens 0,1,2,3,4
query = vocab[2].unsqueeze(0).unsqueeze(0)    # a query vector identical to token 2

out_retrieve, w_retrieve = attention(query, seq, seq)
retrieved_token = out_retrieve[0, 0].argmax().item()

print(f'attention weights over the 5 sequence positions: {w_retrieve[0, 0].round(decimals=3).tolist()}')
print(f'retrieved token id: {retrieved_token}  (queried for token 2)')

plt.figure(figsize=(5, 1.5))
plt.imshow(w_retrieve[0].numpy(), cmap='viridis', aspect='auto')
plt.xlabel('sequence position (key)'); plt.yticks([])
plt.title('Attention weights: query = token 2')
plt.colorbar(fraction=0.02)
plt.show()

Chú ý lấy ra đúng token 2, đánh trọng số vị trí 2 cao hơn hẳn các vị trí khác (được trực quan hóa thành ô sáng trong bản đồ nhiệt). Đây là cơ chế, được lột trần: không có kernel tích chập, không có vùng lân cận cố định — chỉ là "tìm thứ tương tự, lấy trung bình có trọng số nó vào".

## Mặt nạ nhân quả

Một số nhiệm vụ (dự đoán token tiếp theo trong một chuỗi) đòi hỏi một vị trí chỉ có thể chú ý tới các vị trí ở tại hoặc trước nó — chú ý tới tương lai sẽ cho phép mô hình "gian lận" bằng cách nhìn vào đáp án. Điều này được áp đặt bằng một mặt nạ: điểm số cho các vị trí không được phép (tương lai) được đặt thành $-\infty$ trước softmax, để chúng nhận trọng số đúng bằng 0.

In [ ]:
causal_mask = torch.tril(torch.ones(T, T))  # 1 where position i may attend to position j <= i
out_causal, w_causal = attention(Q, K, V, mask=causal_mask)
out_torch_causal = F.scaled_dot_product_attention(Q, K, V, is_causal=True)

future_weight = w_causal[:, 0, 1:].abs().sum().item()  # position 0 attending to positions 1..T-1
print(f'total attention weight from position 0 onto FUTURE positions: {future_weight:.6f}  (should be exactly 0)')
print(f'max abs diff vs F.scaled_dot_product_attention(..., is_causal=True): {(out_causal - out_torch_causal).abs().max().item():.2e}')

# Masked-out (future) positions are literally zero weight, not missing data --
# but displaying them as 0 under viridis paints them dark, which reads as "a
# small but real weight" rather than "disallowed." Replace them with NaN and
# let imshow render NaN as white, so "not attended to at all" looks visually
# distinct from "attended to, but weakly" -- matching the animation below.
w_display = w_causal[0].numpy().copy()
w_display[causal_mask.numpy() == 0] = np.nan
w_display = np.ma.masked_invalid(w_display)
cmap = plt.get_cmap('viridis').copy()
cmap.set_bad('white')

plt.figure(figsize=(4, 3.5))
# vmin=0 anchors the color scale at true zero (matching the animation below),
# rather than letting imshow autoscale to this array's own smallest weight.
plt.imshow(w_display, cmap=cmap, vmin=0, vmax=w_display.max())
plt.xlabel('key position'); plt.ylabel('query position')
plt.title('Causal attention mask\n(every row only attends to itself and the past)')
plt.colorbar(fraction=0.046)
plt.show()

Bản đồ nhiệt tĩnh ở trên cho thấy mặt nạ đã hoàn thành; ảnh động bên dưới cho thấy *tại sao* nó lại có hình tam giác, bằng cách gắn trực tiếp nó với thứ tự sinh — một chuỗi ngắn được tạo ra từng token một, và ở mỗi bước, chỉ hàng chú ý của token mới đó được hiển thị, luôn trên các vị trí từ 0 đến vị trí hiện tại và không bao giờ vượt quá, vì các vị trí sau đơn giản là chưa tồn tại.

<img src="../animations/causal_attention_generation_demo.gif" alt="Ảnh động sinh tự hồi quy một chuỗi token ngắn, từng vị trí một, kết hợp với ma trận trọng số chú ý nhân quả được hiển thị từng hàng: ở mỗi bước, token vừa được sinh được làm nổi bật, các token đã sinh được hiển thị màu xanh lục, các token chưa sinh được làm mờ xám với một dấu chấm hỏi, và ma trận chú ý chỉ hiển thị các hàng cho các vị trí đã sinh, các trọng số khác không của mỗi hàng giới hạn ở các vị trí khóa tại hoặc trước hàng đó, với các hàng chưa hiển thị được che trắng" style="display: block; margin: 0 auto; max-width: 760px;">

Mặt nạ nhân quả (causal masking) chính xác là lý do các mô hình kiểu GPT được gọi là "tự hồi quy (autoregressive)": tại thời điểm sinh, truy vấn của mỗi token mới chỉ có thể thấy khóa và giá trị từ các token đã được sinh, không bao giờ từ các token chưa tồn tại. Các mô hình hai chiều (như BERT, hoặc mọi vision transformer trong bài tiếp theo) bỏ qua hoàn toàn mặt nạ này — một ảnh không có "tương lai", nên mọi mảng đều tự do chú ý tới mọi mảng khác.

Hãy để ý, trong ảnh động ở trên, rằng khóa và giá trị của một hàng đã sinh không bao giờ thay đổi khi các hàng sau được thêm vào — chỉ có hàng mới nhất cần tính khóa và giá trị. Sự thật đó là điều giúp **bộ đệm KV (KV cache)** khả thi tại thời điểm suy luận (Bài 53): thay vì tính lại $K$ và $V$ cho toàn bộ chuỗi ở mỗi bước sinh, một hệ thống thực tế tính khóa và giá trị của mỗi vị trí đúng một lần, lưu đệm chúng, và chỉ nối thêm một cặp khóa/giá trị mới mỗi bước.

### Bài tập

1. Trong ví dụ truy xuất, đổi `query` thành `vocab[2] + vocab[4]` (một sự pha trộn của hai token) thay vì một sự khớp thuần túy. Chú ý giờ có chia trọng số giữa vị trí 2 và 4 không? Sự phân chia trông có gần đều không, và bạn có thể giải thích tại sao từ công thức tích vô hướng không?
2. Bỏ phép co giãn `/ np.sqrt(d_k)` khỏi `attention` và chạy lại ví dụ truy xuất với `D` tăng từ 8 lên 128. Các trọng số chú ý trở nên "nhọn" hơn hay ít hơn (gần one-hot hơn) khi số chiều tăng mà không co giãn — và việc thêm phép co giãn trở lại có khôi phục hành vi ban đầu không?
3. Dựng một mặt nạ cho phép mỗi vị trí chỉ chú ý tới chính nó và các lân cận trực tiếp (một cửa sổ "chú ý cục bộ (local attention)", độ rộng 3) thay vì toàn bộ tam giác nhân quả. Mẫu trọng số thu được so với một kernel tích chập ở Bài 10 áp dụng dọc theo chuỗi thế nào?